In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking and good BC acquisition start
        # df = df[df['Non_Standard_Braking'] == 0]
        # df = df[df['BC_BadStart'] == 0]
        
        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'Dati(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source
        df['Malfunction'] = 0

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue
        
        # --------------------------------------------------------------
        # Additional MBP Phase Classification Error (Monorail only)
        # --------------------------------------------------------------
        if 'Brake_energy_pipe' in df.columns:
            # Ensure numeric
            df['Brake_energy_pipe'] = pd.to_numeric(
                df['Brake_energy_pipe'], errors='coerce'
            )

            # Flag error: 1 if Brake_energy_pipe < 0.01, else 0
            df['MBP_PhaseClassification_error'] = np.where(
                df['Brake_energy_pipe'] < 0.01,
                1,
                0
            )
        else:
            # Column missing → create flag as NaN to avoid silent failure
            df['MBP_PhaseClassification_error'] = np.nan
        
        # --------------------------------------------------------------
        # Additional buildup timing error (Monorail only)
        # --------------------------------------------------------------
        if 'Buildup_timing_pipe' in df.columns:
            # Ensure numeric
            df['Buildup_timing_pipe'] = pd.to_numeric(
                df['Buildup_timing_pipe'], errors='coerce'
            )

            # Flag error: 1 if Buildup_timing_pipe > 180, else 0
            df['MBP_buildup_timing_error'] = np.where(
                df['Buildup_timing_pipe'] > 180,
                1,
                0
            )
        else:
            # Column missing → create flag as NaN to avoid silent failure
            df['MBP_buildup_timing_error'] = np.nan
        
        if 'WV_MeanPressure' in df.columns:
            # Ensure numeric
            df['WV_MeanPressure'] = pd.to_numeric(
                df['WV_MeanPressure'], errors='coerce'
            )

            # Assign levels:
            # < 2  → 0
            # 2–3  → 1
            # > 3  → 2
            df['WV_MeanPressureLevel'] = np.select(
                condlist=[
                    df['WV_MeanPressure'] < 2,
                    (df['WV_MeanPressure'] >= 2) & (df['WV_MeanPressure'] <= 3),
                    df['WV_MeanPressure'] > 3
                ],
                choicelist=[0, 1, 2],
                default=np.nan
            )
        else:
            # Column missing → create flag as NaN to avoid silent failure
            df['WV_MeanPressureLevel'] = np.nan
                
        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base, df_data

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_raw_Dati01.csv',
    'TestBrakefinal_data_raw_Dati06.csv',
    'TestBrakefinal_data_raw_Dati27.csv',
    'TestBrakefinal_data_raw_Dati10.csv',
    'TestBrakefinal_data_raw_Dati11.csv',
    'TestBrakefinal_data_raw_Dati05.csv',
    'TestBrakefinal_data_raw_Dati18.csv',
    'TestBrakefinal_data_raw_Dati32.csv',
    'TestBrakefinal_data_raw_Dati24.csv',
    'TestBrakefinal_data_raw_Dati30.csv'
]

[df, df_monorail] = load_data(model_path, monorail_paths)
df['Malfunction'] = df['Malfunction'].astype(str)
print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

print('df_monorail shape:')
print(df_monorail.shape)

In [ ]:
mapping = {
    1: "T3000",
    6: "T3000",
    27: "T3000",
    30: "T3000",
    10: "4909",
    11: "4909",
    18: "4909",
    24: "4909",
    5: "4575",
    32: "4575"
}
df_monorail["WagonType"] = df_monorail["Source"].map(mapping)
df_monorail.head()

In [ ]:
mapping = {
    1: "HP",
    6: "HP",
    27: "HP",
    30: "LP",
    10: "HP",
    11: "HP",
    18: "HP",
    24: "HP",
    5: "HP",
    32: "HP"
}
df_monorail["Config"] = df_monorail["Source"].map(mapping)
df_monorail.head()

In [ ]:
binary_cols = [
    col for col in df_monorail.columns
    if df_monorail[col].dropna().nunique() <= 2
       and set(df_monorail[col].dropna().unique()).issubset({0, 1})
]

binary_cols

In [ ]:
gps_cols = df_monorail.columns[df_monorail.columns.str.contains("GPS", case=False)].tolist()

gps_cols

In [ ]:
time_cols = df_monorail.columns[df_monorail.columns.str.contains("time", case=False)].tolist()

time_cols

In [ ]:
mbp_cols = df_monorail.columns[df_monorail.columns.str.contains("pipe", case=False)].tolist()

mbp_cols

In [ ]:
df_monorail["Source"].value_counts()

# Initial Data Analysis 

In [ ]:
HEALTH_BINARY_COLS = [
    'BC_AlreadyEngagedStart',
    'BC_BadStart',
    'BC_FlatStartNearZero',
    'BC_PhaseClassification_error',
    'BC_ReleasingAtStart',
    'BC_SensorError',
    'BC_StartAboveThresh',
    'Consecutive_braking_pipe',
    'DS_Error',
    'EmergencyBrake_action',
    'First_phase_error',
    'GPS_SensorError',
    'Gateway_CB_Error',
    'Gateway_VB_Error',
    'MBP_PhaseClassification_error',
    'MBP_Sensor_error',
    'MBP_braketiming_error',
    'Non_Standard_Braking',
    'SV_Error',
    'UB_Error',
    'UR_Error',
    'WV_SensorError',
]


In [ ]:
import pandas as pd

# -------------------------------------------------
# Load data
# -------------------------------------------------
# Restrict to SV_Error == 0
df_monorail = df_monorail[df_monorail["SV_Error"] == 0]

# -------------------------------------------------
# Sanity check
# -------------------------------------------------
assert df_monorail["Non_Standard_Braking"].dropna().isin([0, 1]).all(), \
    "Non_Standard_Braking must be binary (0/1)"

# -------------------------------------------------
# Grouped statistics by WagonType
# -------------------------------------------------
wagon_stats = (
    df_monorail
    .groupby("WagonType")
    .agg(
        Total_Events=("Non_Standard_Braking", "count"),
        NonStandard_Count=("Non_Standard_Braking", "sum"),
    )
)

wagon_stats["Standard_Count"] = (
    wagon_stats["Total_Events"] - wagon_stats["NonStandard_Count"]
)

wagon_stats["NonStandard_Percentage"] = (
    100.0 * wagon_stats["NonStandard_Count"] / wagon_stats["Total_Events"]
)

# Optional: rounding for reporting
wagon_stats["NonStandard_Percentage"] = wagon_stats["NonStandard_Percentage"].round(2)

# Sort for readability (highest anomaly rate first)
wagon_stats = wagon_stats.sort_values(
    by="NonStandard_Percentage",
    ascending=False
)

print(wagon_stats)
wagon_kit_stats = (
    df_monorail
    .groupby(["WagonType", "Source"])
    .agg(
        Total_Events=("Non_Standard_Braking", "count"),
        NonStandard_Count=("Non_Standard_Braking", "sum"),
    )
    .reset_index()
)

wagon_kit_stats["NonStandard_Percentage"] = (
    100.0 * wagon_kit_stats["NonStandard_Count"]
    / wagon_kit_stats["Total_Events"]
).round(2)

print(wagon_kit_stats)



In [ ]:
# -------------------------------------------------
# Load data
# -------------------------------------------------
# Restrict to SV_Error == 0
df_monorail = df_monorail[df_monorail["SV_Error"] == 0]

# -------------------------------------------------
# Sanity check
# -------------------------------------------------
assert df_monorail["EmergencyBrake_action"].dropna().isin([0, 1]).all(), \
    "EmergencyBrake_action must be binary (0/1)"

# -------------------------------------------------
# Grouped statistics by WagonType
# -------------------------------------------------
wagon_stats_eb = (
    df_monorail
    .groupby("WagonType")
    .agg(
        Total_Events=("EmergencyBrake_action", "count"),
        EmergencyBrake_count=("EmergencyBrake_action", "sum"),
    )
)

wagon_stats_eb["ServiceBraking_Count"] = (
    wagon_stats_eb["Total_Events"] - wagon_stats_eb["EmergencyBrake_count"]
)

wagon_stats_eb["EmergencyBrake_Percentage"] = (
    100.0 * wagon_stats_eb["EmergencyBrake_count"] / wagon_stats_eb["Total_Events"]
)

# Optional: rounding for reporting
wagon_stats_eb["EmergencyBrake_Percentage"] = wagon_stats_eb["EmergencyBrake_Percentage"].round(2)

# Sort for readability (highest anomaly rate first)
wagon_stats_eb = wagon_stats_eb.sort_values(
    by="EmergencyBrake_Percentage",
    ascending=False
)

print(wagon_stats_eb)
wagon_kit_stats_eb = (
    df_monorail
    .groupby(["WagonType", "Source"])
    .agg(
        Total_Events=("EmergencyBrake_action", "count"),
        EmergencyBrake_count=("EmergencyBrake_action", "sum"),
    )
    .reset_index()
)

wagon_kit_stats_eb["EmergencyBrake_Percentage"] = (
    100.0 * wagon_kit_stats_eb["EmergencyBrake_count"]
    / wagon_kit_stats_eb["Total_Events"]
).round(2)

print(wagon_kit_stats_eb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_binary_counts_by_wagon_kit(
    df: pd.DataFrame,
    binary_col: str,
    *,
    wagon_type_col: str = "WagonType",
    kit_col: str = "Source",
    sv_error_col: str | None = "SV_Error",
    sv_error_keep: int | None = 0,
    # Naming (required for title/legend)
    name_0: str = "Class 0",
    name_1: str = "Class 1",
    title: str = "Binary Event Counts by Wagon Type and Kit",
    xlabel: str = "WagonType | Kit",
    ylabel: str = "Number of Events",
    # Plot controls
    figsize: tuple = (15, 6),
    rotate_xticks: int = 30,
    annotate_mode: str = "pair_ns",   # "pair_ns" (recommended), "both", "none"
    order_within_wagontype: bool = True,
):
    """
    Create tight side-by-side bar plots (0 vs 1) per WagonType|Kit, using counts
    as bar heights, with optional percentage annotations.

    annotate_mode:
      - "pair_ns": annotate ONLY 1 label per pair above the pair: e.g. "Class1: 28.0%"
                 (recommended to avoid messy labels)
      - "both":    annotate % on BOTH bars (can be messy if one bar is small)
      - "none":    no annotations

    Returns
    -------
    counts : pd.DataFrame
        Table with WagonType, Source, counts for 0/1, Total, and percentage of class 1.
    plot_df : pd.DataFrame
        Long-form table usable for barplot (x=WagonKit, hue=ClassLabel, y=Count).
    fig, ax : matplotlib Figure and Axes
    """

    sns.set_theme(style="whitegrid")

    # -------------------------
    # 0) Filter SV_Error if requested
    # -------------------------
    df_use = df.copy()
    if sv_error_col is not None and sv_error_keep is not None and sv_error_col in df_use.columns:
        df_use = df_use[df_use[sv_error_col] == sv_error_keep].copy()

    # -------------------------
    # 1) Enforce binary (0/1) robustly
    # -------------------------
    if binary_col not in df_use.columns:
        raise KeyError(f"Missing binary column: '{binary_col}'")

    # Convert booleans -> int; else numeric coercion
    if df_use[binary_col].dtype == bool:
        df_use[binary_col] = df_use[binary_col].astype(int)
    else:
        df_use[binary_col] = pd.to_numeric(df_use[binary_col], errors="coerce")

    # Keep only valid {0,1}
    df_use = df_use[df_use[binary_col].isin([0, 1])].copy()

    # -------------------------
    # 2) Build counts table
    # -------------------------
    counts = (
        df_use
        .groupby([wagon_type_col, kit_col])[binary_col]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both 0/1 exist
    for k in [0, 1]:
        if k not in counts.columns:
            counts[k] = 0

    counts = counts.rename(columns={0: f"{name_0}_Count", 1: f"{name_1}_Count"})
    counts["Total_Events"] = counts[f"{name_0}_Count"] + counts[f"{name_1}_Count"]
    counts[f"{name_1}_Percentage"] = np.where(
        counts["Total_Events"] > 0,
        100.0 * counts[f"{name_1}_Count"] / counts["Total_Events"],
        0.0
    ).round(2)

    counts = counts.reset_index()
    counts["WagonKit"] = counts[wagon_type_col].astype(str) + " | " + counts[kit_col].astype(str)

    # -------------------------
    # 3) Long-form plot data
    # -------------------------
    plot_df = counts.melt(
        id_vars=[wagon_type_col, kit_col, "WagonKit", "Total_Events", f"{name_1}_Percentage"],
        value_vars=[f"{name_0}_Count", f"{name_1}_Count"],
        var_name="ClassLabel",
        value_name="Count"
    )

    plot_df["ClassLabel"] = plot_df["ClassLabel"].replace({
        f"{name_0}_Count": f"{name_0} (0)",
        f"{name_1}_Count": f"{name_1} (1)",
    })

    hue_order = [f"{name_0} (0)", f"{name_1} (1)"]

    # -------------------------
    # 4) X order
    # -------------------------
    if order_within_wagontype:
        order = (
            counts.sort_values([wagon_type_col, "Total_Events"], ascending=[True, False])["WagonKit"]
            .tolist()
        )
    else:
        order = (
            counts.sort_values("Total_Events", ascending=False)["WagonKit"]
            .tolist()
        )

    # -------------------------
    # 5) Plot
    # -------------------------
    fig, ax = plt.subplots(figsize=figsize)

    sns.barplot(
        data=plot_df,
        x="WagonKit",
        y="Count",
        hue="ClassLabel",
        hue_order=hue_order,
        order=order,
        ax=ax,
        dodge=True
    )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    ax.margins(x=0.01)
    ax.grid(axis="y", alpha=0.35)
    sns.despine(ax=ax)
    plt.xticks(rotation=rotate_xticks, ha="right")

    # Headroom for labels
    ax.set_ylim(0, ax.get_ylim()[1] * 1.10)

    # -------------------------
    # 6) Annotations (ROBUST): use containers, not patch indexing
    # -------------------------
    if annotate_mode not in ["pair_ns", "both", "none"]:
        raise ValueError("annotate_mode must be one of: 'pair_ns', 'both', 'none'")

    pct_map = counts.set_index("WagonKit")[f"{name_1}_Percentage"].to_dict()

    xlabels = [t.get_text() for t in ax.get_xticklabels()]
    containers = ax.containers  # one per hue in hue_order

    if annotate_mode == "both":
        # annotate % on BOTH bars (may look crowded)
        cont0 = containers[0]  # name_0
        cont1 = containers[1]  # name_1

        for j, bar in enumerate(cont0):
            h = bar.get_height()
            if h <= 0 or j >= len(xlabels): 
                continue
            wk = xlabels[j]
            p1 = pct_map.get(wk, 0.0)
            p0 = 100.0 - p1
            ax.annotate(
                f"{p0:.1f}%",
                (bar.get_x() + bar.get_width()/2, h),
                ha="center", va="bottom", fontsize=9,
                xytext=(0, 3), textcoords="offset points"
            )

        for j, bar in enumerate(cont1):
            h = bar.get_height()
            if h <= 0 or j >= len(xlabels):
                continue
            wk = xlabels[j]
            p1 = pct_map.get(wk, 0.0)
            ax.annotate(
                f"{p1:.1f}%",
                (bar.get_x() + bar.get_width()/2, h),
                ha="center", va="bottom", fontsize=9,
                xytext=(0, 3), textcoords="offset points"
            )

    elif annotate_mode == "pair_ns":
        # annotate only ONE label per pair, above the higher bar: "<name_1>: XX.X%"
        cont0 = containers[0]
        cont1 = containers[1]

        for j in range(min(len(cont0), len(cont1), len(xlabels))):
            bar0 = cont0[j]
            bar1 = cont1[j]

            h0 = bar0.get_height()
            h1 = bar1.get_height()
            if (h0 <= 0) and (h1 <= 0):
                continue

            wk = xlabels[j]
            p1 = pct_map.get(wk, 0.0)

            x_center = (
                (bar0.get_x() + bar0.get_width()/2) +
                (bar1.get_x() + bar1.get_width()/2)
            ) / 2.0

            y_top = max(h0, h1)

            ax.annotate(
                f"{name_1}: {p1:.1f}%",
                (x_center, y_top),
                ha="center", va="bottom", fontsize=9,
                xytext=(0, 6), textcoords="offset points"
            )

    ax.legend(title="", loc="upper right")
    plt.tight_layout()

    return counts, plot_df, fig, ax


In [ ]:
df_wv1 = df_monorail.loc[
    df_monorail['WV_MeanPressureLevel'].eq(1)
].copy()

# fig setting

In [ ]:
plt.rcParams.update({
    'axes.titlesize': 26,
    'axes.labelsize': 24,
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white'
})

In [ ]:
counts_ns, plot_ns, fig, ax = plot_binary_counts_by_wagon_kit(
    df_monorail,
    binary_col="BC_BadStart",
    sv_error_col= "SV_Error",
    sv_error_keep= 0,
    name_0="Standard",
    name_1="Non-Standard",
    title="Percentage of Irregular Start of Braking Action on Each Wagon on All Operating Conditions",
    annotate_mode="both"  # clean: one label per pair
)

In [ ]:
counts_ns, plot_ns, fig, ax = plot_binary_counts_by_wagon_kit(
    df_monorail,
    binary_col="Non_Standard_Braking",
    sv_error_col= "SV_Error",
    sv_error_keep= 0,
    name_0="Standard",
    name_1="Non-Standard",
    title="Percentage of Non Standard Braking Action on Each Wagon on All Operating Conditions",
    annotate_mode="both"  # clean: one label per pair
)


In [ ]:
counts_em, plot_em, fig, ax = plot_binary_counts_by_wagon_kit(
    df_monorail,
    binary_col="EmergencyBrake_action",
    sv_error_col= "SV_Error",
    sv_error_keep= 0,
    name_0="Service Braking",
    name_1="Emergency Braking",
    title="Braking Event Counts by Wagon Type and Kit (Service vs Emergency)",
    annotate_mode="both"
)


In [ ]:
counts_em, plot_em, fig, ax = plot_binary_counts_by_wagon_kit(
    df_monorail,
    binary_col="BC_SensorError",
    sv_error_col= "SV_Error",
    sv_error_keep= 0,
    name_0="Healthy BC Sensor",
    name_1="BC Sensor Error",
    title="Overall BC Sensor Error Counts by Wagon Type and Kit",
    annotate_mode="both"
)


In [ ]:
df_healthy = df_monorail[df_monorail["BC_SensorError"] == 0]
counts_em, plot_em, fig, ax = plot_binary_counts_by_wagon_kit(
    df_healthy,
    binary_col="BC_PhaseClassification_error",
    sv_error_col= "SV_Error",
    sv_error_keep= 0,
    name_0="Usable Phase Classification",
    name_1="Phase Classification Error",
    title="Phase Classification Error Counts (Healthy BC Sensor) by Wagon Type and Kit",
    annotate_mode="both"
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_binary_percent_by_wagontype_stacked(
    df: pd.DataFrame,
    binary_col: str,
    *,
    wagon_type_col: str = "WagonType",
    sv_error_col: str | None = "SV_Error",
    sv_error_keep: int | None = 0,
    name_0: str = "Service Braking (0)",
    name_1: str = "Emergency Braking (1)",
    title: str = "Emergency Braking Percentage of each Wagon Type",
    xlabel: str = "Wagon Type",
    ylabel: str = "Percentage of Events (%)",
    figsize: tuple = (10, 6),
    order_by_total: bool = True,
    # --- label controls ---
    show_labels: bool = True,
    label_fmt: str = "{p1:.1f}%",
    label_inside_threshold: float = 3.0,  # if emergency >= 3%, write inside orange
    # --- legend controls ---
    legend_outside: bool = True,
):
    sns.set_theme(style="whitegrid")

    df_use = df.copy()
    if sv_error_col is not None and sv_error_keep is not None and sv_error_col in df_use.columns:
        df_use = df_use[df_use[sv_error_col] == sv_error_keep].copy()

    if binary_col not in df_use.columns:
        raise KeyError(f"Missing binary column: '{binary_col}'")

    s = df_use[binary_col]
    if s.dtype == bool:
        df_use[binary_col] = s.astype(int)
    else:
        df_use[binary_col] = pd.to_numeric(s, errors="coerce")

    df_use = df_use[df_use[binary_col].isin([0, 1])].copy()

    counts = (
        df_use.groupby(wagon_type_col)[binary_col]
        .value_counts()
        .unstack(fill_value=0)
    )
    if 0 not in counts.columns: counts[0] = 0
    if 1 not in counts.columns: counts[1] = 0

    counts = counts.rename(columns={0: "Count0", 1: "Count1"})
    counts["Total"] = counts["Count0"] + counts["Count1"]
    counts["Pct1"] = np.where(counts["Total"] > 0, 100.0 * counts["Count1"] / counts["Total"], 0.0)
    counts["Pct0"] = 100.0 - counts["Pct1"]
    counts = counts.reset_index()

    if order_by_total:
        counts = counts.sort_values("Total", ascending=False)
    else:
        counts = counts.sort_values(wagon_type_col)

    x = counts[wagon_type_col].astype(str).tolist()
    p0 = counts["Pct0"].to_numpy()
    p1 = counts["Pct1"].to_numpy()

    fig, ax = plt.subplots(figsize=figsize)

    bars0 = ax.bar(x, p0, label=name_0)
    bars1 = ax.bar(x, p1, bottom=p0, label=name_1)

    ax.set_title(title, fontsize=18, pad=30)

    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_ylim(0, 105)  # extra headroom for labels above bars
    ax.grid(axis="y", alpha=0.35)
    sns.despine(ax=ax)

    # ---- labels (better placement) ----
    if show_labels:
        for rect1, base, val1 in zip(bars1, p0, p1):
            xc = rect1.get_x() + rect1.get_width() / 2

            if val1 >= label_inside_threshold:
                # place inside the orange slice
                y = base + val1 / 2
                ax.text(
                    xc, y,
                    label_fmt.format(p1=val1),
                    ha="center", va="center", fontsize=9, color="black"
                )
            else:
                # place just above the bar top, but within ylim headroom
                y = base + val1 + 1.0
                ax.text(
                    xc, y,
                    label_fmt.format(p1=val1),
                    ha="center", va="bottom", fontsize=9
                )

    # ---- legend placement ----
    ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.08),   # centered, just below title
    ncol=2,                       # horizontal legend
    frameon=False
)


    return counts, fig, ax


Function to plot figure

In [ ]:
def plot_binary_percent_by_wagon_kit(
    df: pd.DataFrame,
    binary_col: str,
    *,
    wagon_type_col: str = "WagonType",
    kit_col: str = "Source",
    sv_error_col: str | None = "SV_Error",
    sv_error_keep: int | None = 0,
    # Naming
    name_0: str = "Class 0",
    name_1: str = "Class 1",
    title: str = "Binary Event Percentage by Wagon Type and Kit",
    xlabel: str = "WagonType | Kit",
    ylabel: str = "Percentage of Events (%)",
    # Plot controls
    figsize: tuple = (15, 6),
    rotate_xticks: int = 30,
    order_within_wagontype: bool = True,
):
    """
    Side-by-side bar plots (0 vs 1) per WagonType|Kit,
    with Y-axis showing PERCENTAGE (0–100), no counts.
    """

    sns.set_theme(style="whitegrid")

    # -------------------------
    # 0) Filter SV_Error if requested
    # -------------------------
    df_use = df.copy()
    if sv_error_col is not None and sv_error_keep is not None and sv_error_col in df_use.columns:
        df_use = df_use[df_use[sv_error_col] == sv_error_keep].copy()

    # -------------------------
    # 1) Enforce binary {0,1}
    # -------------------------
    if binary_col not in df_use.columns:
        raise KeyError(f"Missing binary column: '{binary_col}'")

    if df_use[binary_col].dtype == bool:
        df_use[binary_col] = df_use[binary_col].astype(int)
    else:
        df_use[binary_col] = pd.to_numeric(df_use[binary_col], errors="coerce")

    df_use = df_use[df_use[binary_col].isin([0, 1])].copy()

    # -------------------------
    # 2) Counts per WagonType|Kit
    # -------------------------
    counts = (
        df_use
        .groupby([wagon_type_col, kit_col])[binary_col]
        .value_counts()
        .unstack(fill_value=0)
    )

    if 0 not in counts.columns: counts[0] = 0
    if 1 not in counts.columns: counts[1] = 0

    counts["Total"] = counts[0] + counts[1]

    counts[f"{name_1}_Pct"] = np.where(
        counts["Total"] > 0,
        100.0 * counts[1] / counts["Total"],
        0.0
    )
    counts[f"{name_0}_Pct"] = 100.0 - counts[f"{name_1}_Pct"]

    counts = counts.reset_index()
    counts["WagonKit"] = counts[wagon_type_col].astype(str) + " | " + counts[kit_col].astype(str)

    # -------------------------
    # 3) Long-form (percentage)
    # -------------------------
    plot_df = counts.melt(
        id_vars=[wagon_type_col, kit_col, "WagonKit"],
        value_vars=[f"{name_0}_Pct", f"{name_1}_Pct"],
        var_name="ClassLabel",
        value_name="Percentage"
    )

    plot_df["ClassLabel"] = plot_df["ClassLabel"].replace({
        f"{name_0}_Pct": f"{name_0} (0)",
        f"{name_1}_Pct": f"{name_1} (1)",
    })

    hue_order = [f"{name_0} (0)", f"{name_1} (1)"]

    # -------------------------
    # 4) X ordering
    # -------------------------
    if order_within_wagontype:
        order = (
            counts.sort_values([wagon_type_col, "Total"], ascending=[True, False])["WagonKit"]
            .tolist()
        )
    else:
        order = counts.sort_values("Total", ascending=False)["WagonKit"].tolist()

    # -------------------------
    # 5) Plot
    # -------------------------
    fig, ax = plt.subplots(figsize=figsize)

    sns.barplot(
        data=plot_df,
        x="WagonKit",
        y="Percentage",
        hue="ClassLabel",
        hue_order=hue_order,
        order=order,
        ax=ax,
        dodge=True
    )

    ax.set_title(title, fontsize=26, pad=20)
    ax.set_xlabel(xlabel,fontsize=20)
    ax.set_ylabel(ylabel,fontsize=20)

    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.35)
    sns.despine(ax=ax)

    plt.xticks(rotation=rotate_xticks, ha="right",fontsize=16)
    ax.legend(title="", loc="upper right", fontsize=16)

    plt.tight_layout()

    return counts, plot_df, fig, ax


In [ ]:
plot_binary_percent_by_wagon_kit(
    df_monorail,
    binary_col="EmergencyBrake_action",
    wagon_type_col="WagonType",
    kit_col="Source",
    sv_error_col="SV_Error",
    sv_error_keep=0,
    name_0="Service Braking",
    name_1="Emergency Braking",
    title="Percentage of Emergency Braking Action for Each Monitored Wagon",
    xlabel="WagonType | Kit",
    ylabel="Percentage of Events (%)",
    figsize=(15, 6),
    rotate_xticks=30,
    order_within_wagontype=True,
)

In [ ]:
plot_binary_percent_by_wagon_kit(
    df_monorail,
    binary_col="Non_Standard_Braking",
    wagon_type_col="WagonType",
    kit_col="Source",
    sv_error_col="SV_Error",
    sv_error_keep=0,
    name_0="Standard Braking",
    name_1="Non Standard Braking",
    title="Percentage of Non Standard Braking Action on Each Monitored Wagon",
    xlabel="WagonType | Kit",
    ylabel="Percentage of Events (%)",
    figsize=(15, 6),
    rotate_xticks=30,
    order_within_wagontype=True,
)

In [ ]:
plot_binary_percent_by_wagon_kit(
    df_monorail,
    binary_col="BC_BadStart",
    wagon_type_col="WagonType",
    kit_col="Source",
    sv_error_col="SV_Error",
    sv_error_keep=0,
    name_0="Regular Braking Start",
    name_1="Irregular Braking Start",
    title="Percentage of Non Standard Braking Action on Each Monitored Wagon",
    xlabel="WagonType | Kit",
    ylabel="Percentage of Events (%)",
    figsize=(15, 6),
    rotate_xticks=30,
    order_within_wagontype=True,
)

In [ ]:
df_phaseerror = df_monorail.loc[
    df_monorail['BC_PhaseClassification_error'].eq(1)
].copy()

print(df_phaseerror.shape)
df_phaseerror.head()

In [ ]:
df_nonstandard = df_monorail.loc[
    df_monorail['Non_Standard_Braking'].eq(1) &
    df_monorail['WV_MeanPressureLevel'].eq(1)
].copy()

print(df_nonstandard.shape)
df_nonstandard.head()

In [ ]:
df_phaseerror_SV = df_monorail.loc[
    (df_monorail['BC_PhaseClassification_error'].eq(1)) &
    (df_monorail['SV_Error'].eq(0))
].copy()

print(df_phaseerror_SV.shape)
df_phaseerror_SV.head()

In [ ]:
df_standard = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0)) &
    (df_monorail['Source'].isin([1, 6, 27]))
    
].copy()

print(df_standard.shape)
df_standard.head()

In [ ]:
df_standard["SV_Error"].value_counts()

In [ ]:
df_T3000base = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['Source'].isin([1,6,27])) &
    (df_monorail['SV_Error'].eq(0))
].copy()

print(df_T3000base.shape)
df_T3000base.head()

In [ ]:
df_kit30 = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['WV_MeanPressureLevel'].eq(1)) &
    (df_monorail['BC_PhaseClassification_error'].eq(0)) &
    (df_monorail['MBP_PhaseClassification_error'].eq(0)) &
    (df_monorail['Source'].isin([30]))   
].copy()

print(df_kit30.shape)
df_kit30.head()

In [ ]:
df_T3000 = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['WV_MeanPressureLevel'].eq(1)) &
    (df_monorail['Source'].isin([1,6,27]))   
].copy()

print(df_T3000.shape)

# Total samples in the filtered dataset
N = len(df_T3000)
# MBP error percentage
mbp_error_pct = (
    df_T3000["MBP_PhaseClassification_error"]
    .fillna(0)          # or .dropna() depending on your policy
    .mean() * 100
)

# BC error percentage
bc_error_pct = (
    df_T3000["BC_PhaseClassification_error"]
    .fillna(0)
    .mean() * 100
)

print(f"MBP Phase Classification Error Rate: {mbp_error_pct:.2f}%")
print(f"BC Phase Classification Error Rate:  {bc_error_pct:.2f}%")

df_T3000.head()


In [ ]:
df_T3000_lax = df_T3000.loc[
    (df_T3000['BC_BadStart'].eq(0))
].copy()
print(df_T3000_lax.shape)
df_T3000_lax.head()

# Total samples in the filtered dataset
N = len(df_T3000_lax)
# MBP error percentage
mbp_error_pct = (
    df_T3000_lax["MBP_PhaseClassification_error"]
    .fillna(0)          # or .dropna() depending on your policy
    .mean() * 100
)

# BC error percentage
bc_error_pct = (
    df_T3000_lax["BC_PhaseClassification_error"]
    .fillna(0)
    .mean() * 100
)

print(f"MBP Phase Classification Error Rate: {mbp_error_pct:.2f}%")
print(f"BC Phase Classification Error Rate:  {bc_error_pct:.2f}%")

In [ ]:
df_T3000_hardfilter = df_T3000.loc[
    (df_T3000['BC_PhaseClassification_error'].eq(0)) &
    (df_T3000['MBP_PhaseClassification_error'].eq(0)) &
    (df_T3000['BC_BadStart'].eq(0))
    ].copy()
print(df_T3000_hardfilter.shape)
df_T3000_hardfilter.head()

In [ ]:
df_bchealthy = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_PhaseClassification_error'].eq(0)) &
    (df_monorail['MBP_PhaseClassification_error'].eq(0)) &
    (df_monorail['MBP_buildup_timing_error'].eq(0)) &
    (df_monorail['BC_braketiming_error'].eq(0)) &
    (df_monorail['BC_SensorError'].eq(0))
].copy()

print(df_bchealthy.shape)
df_bchealthy.head()

In [ ]:
print("unique WagonTypes in standard braking set:", df_standard['WagonType'].unique())
print("unique Kit Sources in standard braking set:", df_standard['Source'].unique())

In [ ]:
df_negative_eff = df_T3000_lax.loc[df_T3000_lax['Total_power_efficiency'] < 1]
print(df_negative_eff.shape)
df_negative_eff.head()

In [ ]:
df_negative_brake = df_T3000_lax.loc[df_T3000_lax['Brake_energy_pipe'] < 0.01]
print(df_negative_brake.shape)
df_negative_brake.head()

In [ ]:
df_toohigh_eff = df_T3000_lax.loc[df_T3000_lax['Total_energy_efficiency'] > 200]
print(df_toohigh_eff.shape)
df_toohigh_eff.head()

In [ ]:
df_wv1 = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_PhaseClassification_error'].eq(0)) &
    (df_monorail['MBP_PhaseClassification_error'].eq(0)) &
    (df_monorail['MBP_buildup_timing_error'].eq(0)) &
    (df_monorail['BC_braketiming_error'].eq(0)) &
    (df_monorail['BC_SensorError'].eq(0)) &
    (df_monorail['WV_MeanPressureLevel'].eq(1))
].copy()

print(df_wv1.shape)
df_wv1.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 6))
plt.boxplot(
    df_T3000["WV_MeanPressure"].dropna(),
    vert=True,
    showfliers=True
)

plt.ylabel("WV Mean Pressure [bar]")
plt.title("Distribution of WV Mean Pressure (T3000, Standard Braking)")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Drop NaNs to avoid plotting issues
df_plot = df_T3000.dropna(subset=["WV_MeanPressure", "Source"])

# Sort wagon types for consistent ordering
wagon_types = sorted(df_plot["Source"].unique())

# Collect data per wagon type
data = [
    df_plot.loc[df_plot["Source"] == wt, "WV_MeanPressure"].values
    for wt in wagon_types
]

plt.figure(figsize=(8, 5))
plt.boxplot(
    data,
    labels=wagon_types,
    showfliers=True
)

plt.xlabel("Wagon kit")
plt.ylabel("WV Mean Pressure [bar]")
plt.title("Distribution of WV Mean Pressure of T3000 Wagon")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt

def boxplot_features_by_source(
    df,
    features,
    source_col="Source",
    wagon_filter=None,      # <--- NEW
    y_ref=None,
    ylim_mode="robust",
    margin=0.10,
    show_fliers=True
):
    # Apply wagon filter if provided
    if wagon_filter is not None:
        df = df[df["WagonType"].isin([wagon_filter])]

    wagon_types = sorted(df["WagonType"].dropna().unique())
    """
    Boxplot selected numeric features (one figure per feature),
    grouped by Source, with dynamic y-limits.

    Parameters
    ----------
    ylim_mode : str
        "robust" → use 1–99 percentiles (recommended)
        "minmax" → use exact min/max
    margin : float
        Extra headroom added to ymax (fraction).
    """
    wagon_types = sorted(df["WagonType"].dropna().unique())

    # Generate distinct colors
    cmap = cm.get_cmap("Set2", len(wagon_types))
    wagon_colors = {wt: mcolors.to_hex(cmap(i)) for i, wt in enumerate(wagon_types)}
    sources = sorted(df[source_col].dropna().unique())

    for feat in features:
        if feat not in df.columns:
            print(f"[SKIP] Feature not found: {feat}")
            continue

        # ---- collect per-source data ----
        data = []
        labels = []
        all_vals = []

        wagon_types = sorted(df["WagonType"].dropna().unique())
        sources = sorted(df[source_col].dropna().unique())

        box_wagon_types = []   # parallel list to `data`

        for wt in wagon_types:
            for src in sources:
                mask = (df["WagonType"] == wt) & (df[source_col] == src)
                vals = df.loc[mask, feat]
                vals = vals.replace(r"^\s*$", np.nan, regex=True).dropna()

                if not vals.empty:
                    data.append(vals.values)
                    labels.append(f"{wt} — {src}")
                    all_vals.append(vals.values)
                    box_wagon_types.append(wt)

        if not data:
            print(f"[SKIP] No valid data for feature: {feat}")
            continue

        all_vals = np.concatenate(all_vals)

        # ---- dynamic y-limits ----
        if ylim_mode == "robust":
            y_low = np.nanpercentile(all_vals, 1)
            y_high = np.nanpercentile(all_vals, 99)
        else:  # "minmax"
            y_low = min(0,np.nanmin(all_vals))
            y_high = np.nanmax(all_vals)

        if y_high <= y_low:
            y_high = y_low + 1.0

        y_high *= (1 + margin)

        # ---- plotting ----
        fig, ax = plt.subplots(figsize=(max(10, 0.35 * len(labels)), 5))

        bp = ax.boxplot(
            data,
            labels=labels,
            patch_artist=True,
            showfliers=show_fliers,
            medianprops=dict(color="black", linewidth=1.5)
        )

        for box, wt in zip(bp["boxes"], box_wagon_types):
            box.set_facecolor(wagon_colors[wt])
            box.set_alpha(0.85)

        if y_ref is not None:
            ax.axhline(y_ref, linestyle="--", color="black", linewidth=1.2)

        ax.set_title(f"{feat} — Monorail Kit", fontsize=13)
        ax.set_xlabel("Source")
        ax.set_ylabel(feat)
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_ylim(y_low, y_high)

        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


In [ ]:
features_to_explore = [
    "Total_power_efficiency",
    "Total_power_delay",
    "Std_delay_exp",
    "WV_MeanPressure",

]


boxplot_features_by_source(
    df_T3000_lax,
    features=features_to_explore,
    source_col="Source",
    ylim_mode="minmax",   # ← best for your dataset
    margin=0.15
)


In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt

def boxplot_features_by_source_v2(
    df,
    features,
    source_col="Source",
    wagon_filter=None,
    y_ref=None,
    ylim_mode="robust",
    margin=0.10,
    show_fliers=True,
    source_30_color="tab:blue"   # <--- NEW
):
    # Apply wagon filter if provided
    if wagon_filter is not None:
        df = df[df["WagonType"].isin([wagon_filter])]

    # Generate distinct colors per WagonType (same as your code)
    wagon_types = sorted(df["WagonType"].dropna().unique())
    cmap = cm.get_cmap("Set2", len(wagon_types))
    wagon_colors = {wt: mcolors.to_hex(cmap(i)) for i, wt in enumerate(wagon_types)}

    sources = sorted(df[source_col].dropna().unique())

    for feat in features:
        if feat not in df.columns:
            print(f"[SKIP] Feature not found: {feat}")
            continue

        data = []
        labels = []
        all_vals = []

        wagon_types = sorted(df["WagonType"].dropna().unique())
        sources = sorted(df[source_col].dropna().unique())

        box_wagon_types = []   # parallel list to data
        box_sources     = []   # <--- NEW: parallel list to data

        for wt in wagon_types:
            for src in sources:
                mask = (df["WagonType"] == wt) & (df[source_col] == src)
                vals = df.loc[mask, feat]
                vals = vals.replace(r"^\s*$", np.nan, regex=True).dropna()

                if not vals.empty:
                    data.append(vals.values)
                    labels.append(f"{wt} — {src}")
                    all_vals.append(vals.values)
                    box_wagon_types.append(wt)
                    box_sources.append(src)  # <--- NEW

        if not data:
            print(f"[SKIP] No valid data for feature: {feat}")
            continue

        all_vals = np.concatenate(all_vals)

        # ---- dynamic y-limits (unchanged) ----
        if ylim_mode == "robust":
            y_low = np.nanpercentile(all_vals, 1)
            y_high = np.nanpercentile(all_vals, 99)
        else:  # "minmax"
            y_low = min(0, np.nanmin(all_vals))
            y_high = np.nanmax(all_vals)

        if y_high <= y_low:
            y_high = y_low + 1.0

        y_high *= (1 + margin)

        # ---- plotting ----
        fig, ax = plt.subplots(figsize=(max(10, 0.35 * len(labels)), 5))

        bp = ax.boxplot(
            data,
            labels=labels,
            patch_artist=True,
            showfliers=show_fliers,
            medianprops=dict(color="black", linewidth=1.5)
        )

        # Color boxes: default by WagonType, but override Source==30 to blue
        for box, wt, src in zip(bp["boxes"], box_wagon_types, box_sources):
            if src == 30:
                box.set_facecolor(source_30_color)  # <--- hard-coded Source=30
            else:
                box.set_facecolor(wagon_colors[wt]) # default behavior

            box.set_alpha(0.85)

        if y_ref is not None:
            ax.axhline(y_ref, linestyle="--", color="black", linewidth=1.2)

        ax.set_title(f"{feat} — Monorail Kit", fontsize=13)
        ax.set_xlabel("Source")
        ax.set_ylabel(feat)
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_ylim(y_low, y_high)

        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
plt.rcParams.update({
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white'
})
def boxplot_T3000_HP_LP(
    df,
    features,
    wagon_type="T3000",
    config_col="Config",
    wagon_col="WagonType",
    configs=("HP", "LP"),
    y_ref=None,
    ylim_mode=None,
    margin=0.10,
    show_fliers=True,
):
    # 1) Filter: WagonType == T3000 and Config in {HP, LP}
    dff = df.loc[
        (df[wagon_col].astype(str) == str(wagon_type)) &
        (df[config_col].isin(list(configs)))
    ].copy()

    # Ensure consistent order: HP then LP (if present)
    configs_present = [c for c in configs if c in dff[config_col].unique()]

    if len(configs_present) == 0:
        print(f"[SKIP] No rows found for WagonType={wagon_type} and Config in {configs}.")
        return

    # Simple two-color palette
    cmap = cm.get_cmap("Set2", 2)
    config_colors = {
        configs_present[i]: mcolors.to_hex(cmap(i)) for i in range(len(configs_present))
    }

    for feat in features:
        if feat not in dff.columns:
            print(f"[SKIP] Feature not found: {feat}")
            continue

        # 2) Build the two groups (HP / LP)
        data = []
        labels = []
        all_vals = []

        for cfg in configs_present:
            vals = dff.loc[dff[config_col] == cfg, feat]
            vals = vals.replace(r"^\s*$", np.nan, regex=True)
            vals = np.array(vals, dtype="float64")
            vals = vals[~np.isnan(vals)]

            if vals.size > 0:
                data.append(vals)
                labels.append(cfg)
                all_vals.append(vals)

        if not data:
            print(f"[SKIP] No valid numeric data for feature: {feat}")
            continue

        all_vals = np.concatenate(all_vals)

        # 3) Dynamic y-limits
        if ylim_mode == "robust":
            y_low = np.nanpercentile(all_vals, 1)
            y_high = np.nanpercentile(all_vals, 99)
        else:
            y_low = min(0, np.nanmin(all_vals))
            y_high = np.nanmax(all_vals)

        if y_high <= y_low:
            y_high = y_low + 1.0
        y_high *= (1 + margin)

        # 4) Plot
        fig, ax = plt.subplots(figsize=(6.5, 4.8))

        bp = ax.boxplot(
            data,
            labels=labels,
            patch_artist=True,
            showfliers=show_fliers,
            medianprops=dict(color="black", linewidth=1.5),
        )

        for box, cfg in zip(bp["boxes"], labels):
            box.set_facecolor(config_colors.get(cfg, "lightgray"))
            box.set_alpha(0.85)

        if y_ref is not None:
            ax.axhline(y_ref, linestyle="--", color="black", linewidth=1.2)

        ax.set_title(f"{feat} — {wagon_type} (HP vs LP)")
        ax.set_xlabel("Wagon Configuration")
        ax.set_ylabel(feat)
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_ylim(y_low, y_high)

        plt.tight_layout()
        plt.show()


# Manual Brake Features

In [ ]:
mask = df_monorail["Source"] == 30
unique_ids_30 = df_monorail.loc[mask, "WV_ID"].unique()
unique_ids_30

In [ ]:
# From df_h select only the row with WV_ID that is SD, 0x43, 0x9d, 0xfc
# Define WV_ID values correlated to central bogie and San Donato
allowed_wv_ids = ["0x43", "0x9d", "0xfc",'0xc', '0xd1', '0x4b']

df_T3000_mb = df_monorail.loc[
    (df_monorail['WV_ID'].isin(allowed_wv_ids)) &
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0)) &
    (df_monorail['SV_Error'].eq(0)) &
    (df_monorail['First_phase_error'].eq(0)) &
    (df_monorail['WV_MeanPressureLevel'].isin([1, 2]))
].copy()

features_to_explore = [
    "First_phase_mean_curvature",
    "First_phase_half_time_ratio",
]

print(df_T3000_mb.shape)
df_T3000_mb.head()

boxplot_T3000_HP_LP(
    df_T3000_mb,
    features=features_to_explore,
    wagon_type="T3000",
    configs=("HP", "LP"),
)


In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt

def boxplot_features_by_source_mpl(
    df,
    features,
    source_col="Source",
    wagon_filter=None,
    y_ref=None,
    ylim_mode="robust",
    margin=0.10,
    show_fliers=True,
    # ---- NEW controls ----
    xlabel="Source",
    ylabel=None,                 # if None -> use feature name
    title_fmt="{feat} — Monorail",
    title_fontsize=13,
    label_fontsize=12,
    tick_fontsize=10,
    xtick_rotation=45,
    fig_height=5.0,
    fig_width_per_box=0.35,
    min_fig_width=10.0,
):
    """
    Matplotlib boxplots of selected numeric features grouped by WagonType × source_col.

    New controls:
    - xlabel, ylabel, title_fmt
    - title_fontsize, label_fontsize, tick_fontsize
    - xtick_rotation
    - figure sizing controls

    Notes:
    - Colors encode WagonType (Set2 colormap).
    - One figure per feature.
    """

    df = df.copy()

    # Apply wagon filter if provided
    if wagon_filter is not None:
        df = df[df["WagonType"].isin([wagon_filter])]

    wagon_types = sorted(df["WagonType"].dropna().unique())
    sources = sorted(df[source_col].dropna().unique())

    if len(wagon_types) == 0 or len(sources) == 0:
        print("[SKIP] No wagon types or sources available after filtering.")
        return

    # Generate distinct colors for wagon types
    cmap = cm.get_cmap("Set2", len(wagon_types))
    wagon_colors = {wt: mcolors.to_hex(cmap(i)) for i, wt in enumerate(wagon_types)}

    for feat in features:
        if feat not in df.columns:
            print(f"[SKIP] Feature not found: {feat}")
            continue

        # ---- collect per-group data ----
        data = []
        labels = []
        all_vals = []
        box_wagon_types = []

        for wt in wagon_types:
            for src in sources:
                mask = (df["WagonType"] == wt) & (df[source_col] == src)
                vals = df.loc[mask, feat]

                # coerce numeric safely
                vals = vals.replace(r"^\s*$", np.nan, regex=True)
                vals = pd.to_numeric(vals, errors="coerce")
                vals = vals.dropna()

                if not vals.empty:
                    data.append(vals.values)
                    labels.append(f"{wt}")
                    all_vals.append(vals.values)
                    box_wagon_types.append(wt)

        if not data:
            print(f"[SKIP] No valid data for feature: {feat}")
            continue

        all_vals = np.concatenate(all_vals)

        # ---- dynamic y-limits ----
        if ylim_mode == "robust":
            y_low = np.nanpercentile(all_vals, 1)
            y_high = np.nanpercentile(all_vals, 99)
        else:  # "minmax"
            y_low = min(0, float(np.nanmin(all_vals)))
            y_high = float(np.nanmax(all_vals))

        if y_high <= y_low:
            y_high = y_low + 1.0

        y_high = y_high * (1 + margin)

        # ---- figure size ----
        fig_w = max(min_fig_width, fig_width_per_box * len(labels))
        fig, ax = plt.subplots(figsize=(fig_w, fig_height))

        bp = ax.boxplot(
            data,
            labels=labels,
            patch_artist=True,
            showfliers=show_fliers,
            medianprops=dict(color="black", linewidth=1.5)
        )

        for box, wt in zip(bp["boxes"], box_wagon_types):
            box.set_facecolor(wagon_colors[wt])
            box.set_alpha(0.85)

        if y_ref is not None:
            ax.axhline(y_ref, linestyle="--", color="black", linewidth=1.2)

        # ---- labels + fonts ----
        title = title_fmt.format(feat=feat)
        ax.set_title(title, fontsize=title_fontsize)

        ax.set_xlabel(xlabel, fontsize=label_fontsize)
        y_label_text = feat if ylabel is None else ylabel
        ax.set_ylabel(f"{y_label_text} [bar]", fontsize=label_fontsize)

        ax.grid(True, axis="y", alpha=0.3)
        ax.set_ylim(0, 6)

        ax.tick_params(axis="both", labelsize=tick_fontsize)
        plt.xticks(rotation=xtick_rotation, ha="right")
        ax.set_xticks(np.arange(1, len(labels) + 1))
        ax.set_xticklabels(labels, ha="center")

        ax.set_xlabel("Wagon Type", fontsize=label_fontsize, labelpad=12)

        plt.tight_layout()
        plt.show()


In [ ]:
df_filter = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0))
].copy()

features_to_explore = [
    "WV_MeanPressure",
    "Total_power_efficiency",
]

boxplot_features_by_source_mpl(
    df=df_filter,
    features=features_to_explore,
    source_col="WagonType",
    xlabel="Wagon Type",
    title_fontsize=20,
    label_fontsize=18,
    tick_fontsize=16,
    xtick_rotation=0
)



In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

def boxplot_features_by_source_plotly(
    df,
    features,
    source_col="Source",
    wagon_filter=None,
    y_ref=None,                 # Plotly: will add a dashed line if provided
    ylim_mode="robust",
    margin=0.10,
    show_fliers=True,
    # ---- NEW controls ----
    xlabel="Group",
    ylabel=None,                # if None -> use feature name
    title_fmt="{feat} — Monorail Kit",
    x_order=None,               # optional explicit order for x categories
    height=500,
    width=None,
):
    """
    Plotly interactive boxplots, grouped by WagonType × source_col.

    - Colors: WagonType (legend).
    - X: combined label "WagonType — Source" by default.
    - One figure per feature.
    """

    df = df.copy()

    if wagon_filter is not None:
        df = df[df["WagonType"].isin([wagon_filter])]

    if df.empty:
        print("[SKIP] Empty dataframe after filtering.")
        return

    # Ensure group columns are strings
    df["WagonType"] = df["WagonType"].astype(str)
    df[source_col] = df[source_col].astype(str)

    # Combined group label on x-axis
    df["_Group"] = df["WagonType"] + " — " + df[source_col]

    for feat in features:
        if feat not in df.columns:
            print(f"[SKIP] Feature not found: {feat}")
            continue

        # numeric conversion
        tmp = df[["_Group", "WagonType", source_col, feat]].copy()
        tmp[feat] = pd.to_numeric(tmp[feat], errors="coerce")
        tmp = tmp.dropna(subset=[feat])

        if tmp.empty:
            print(f"[SKIP] No valid data for feature: {feat}")
            continue

        vals = tmp[feat].to_numpy()

        # y-limits
        if ylim_mode == "robust":
            y_low = float(np.nanpercentile(vals, 1))
            y_high = float(np.nanpercentile(vals, 99))
        else:
            y_low = min(0.0, float(np.nanmin(vals)))
            y_high = float(np.nanmax(vals))

        if y_high <= y_low:
            y_high = y_low + 1.0
        y_high = y_high * (1 + margin)

        # Plotly box: points="outliers" vs False
        points = "outliers" if show_fliers else False

        fig = px.box(
            tmp,
            x="_Group",
            y=feat,
            color="WagonType",
            points=points,
            title=title_fmt.format(feat=feat),
            category_orders={"_Group": x_order} if x_order is not None else None,
        )

        fig.update_layout(
            xaxis_title=xlabel,
            yaxis_title=(feat if ylabel is None else ylabel),
            height=height,
            width=width,
            margin=dict(l=40, r=20, t=60, b=120),
            legend_title_text="WagonType",
        )

        fig.update_xaxes(tickangle=45)

        # Fix y range
        fig.update_yaxes(range=[y_low, y_high])

        # Add reference horizontal line if requested
        if y_ref is not None:
            fig.add_hline(
                y=y_ref,
                line_dash="dash",
                line_width=2
            )

        fig.show()


In [ ]:
features_to_explore = [
    "WV_MeanPressure",
    "Total_power_efficiency",
    "Total_power_delay",
    "Std_delay_exp",
]

boxplot_features_by_source_plotly(
    df=df_filter,
    features=features_to_explore,
    source_col="Source",
    xlabel="WagonType — Source",
    height=550,
    show_fliers=True
)


In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def boxplot_features_by_source(
    df,
    features,
    source_col="Source",
    wagon_filter=None,
    y_ref=None,
    ylim_mode="robust",
    margin=0.10,
    show_fliers=True,
    # --- NEW ---
    separate_by_config=False,
    config_col="Config",
    configs=("HP", "LP"),
    config_colors=None,   # optional dict {"HP": "...", "LP": "..."}
):
    """
    Boxplot numeric features grouped by WagonType × source_col.
    Optionally split each group by Config (HP/LP) into side-by-side boxes.

    Notes:
    - Box facecolor encodes WagonType (as before).
    - If separate_by_config=True, box edgecolor encodes Config (HP/LP),
      so we keep WagonType coloring while still showing Config separation.
    """

    df = df.copy()

    # Apply wagon filter if provided
    if wagon_filter is not None:
        df = df[df["WagonType"].isin([wagon_filter])]

    wagon_types = sorted(df["WagonType"].dropna().unique())
    sources = sorted(df[source_col].dropna().unique())

    # WagonType colors (same as your approach)
    cmap = cm.get_cmap("Set2", max(len(wagon_types), 1))
    wagon_colors = {wt: mcolors.to_hex(cmap(i)) for i, wt in enumerate(wagon_types)}

    # Config colors (only used in legend/edges)
    if config_colors is None:
        config_colors = {configs[0]: "#1f77b4", configs[1]: "#d62728"}  # blue/red edges

    for feat in features:
        if feat not in df.columns:
            print(f"[SKIP] Feature not found: {feat}")
            continue

        data = []
        labels = []
        all_vals = []
        box_wagon_types = []
        box_configs = []  # NEW: parallel to data

        # ---- collect data ----
        for wt in wagon_types:
            for src in sources:
                base_mask = (df["WagonType"] == wt) & (df[source_col] == src)

                if separate_by_config:
                    # create two boxes per group: HP and LP (if present)
                    for cfg in configs:
                        mask = base_mask & (df[config_col] == cfg)
                        vals = df.loc[mask, feat]
                        vals = vals.replace(r"^\s*$", np.nan, regex=True).dropna()

                        if not vals.empty:
                            data.append(vals.values)
                            labels.append(f"{wt} — {src}")  # keep same x-category text
                            all_vals.append(vals.values)
                            box_wagon_types.append(wt)
                            box_configs.append(cfg)
                else:
                    # original behavior: one box per group
                    vals = df.loc[base_mask, feat]
                    vals = vals.replace(r"^\s*$", np.nan, regex=True).dropna()

                    if not vals.empty:
                        data.append(vals.values)
                        labels.append(f"{wt} — {src}")
                        all_vals.append(vals.values)
                        box_wagon_types.append(wt)

                        # detect config composition of this group
                        cfgs = df.loc[base_mask, config_col].dropna().unique()
                        box_configs.append("LP" if (len(cfgs) == 1 and cfgs[0] == "LP") else "HP")

        if not data:
            print(f"[SKIP] No valid data for feature: {feat}")
            continue

        all_vals = np.concatenate(all_vals)

        # ---- dynamic y-limits ----
        if ylim_mode == "robust":
            y_low = np.nanpercentile(all_vals, 1)
            y_high = np.nanpercentile(all_vals, 99)
        else:
            y_low = min(0, np.nanmin(all_vals))
            y_high = np.nanmax(all_vals)

        if y_high <= y_low:
            y_high = y_low + 1.0
        y_high *= (1 + margin)

        # ---- plotting ----
        fig, ax = plt.subplots(figsize=(max(10, 0.35 * len(labels)), 5))

        # If we split by config, we need explicit positions to get side-by-side boxes
        if separate_by_config:
            # group indices correspond to unique label entries in order of appearance
            # We keep the same text label for HP/LP, but shift positions slightly.
            n = len(data)
            base_pos = np.arange(1, n + 1, dtype=float)

            # alternate offsets: first cfg -> left, second cfg -> right
            # This assumes configs has length 2 (HP/LP).
            offsets = {configs[0]: -0.18, configs[1]: +0.18}
            positions = np.array([base_pos[i] + offsets.get(box_configs[i], 0.0) for i in range(n)])

            bp = ax.boxplot(
                data,
                positions=positions,
                widths=0.32,
                patch_artist=True,
                showfliers=show_fliers,
                medianprops=dict(color="black", linewidth=1.5)
            )

            # Place xticks at the center of each “pair” (every two entries belonging to same wt-src)
            # Because you reuse the same label text, we set ticks on base_pos and show labels there.
            ax.set_xticks(base_pos)
            ax.set_xticklabels(labels)
        else:
            bp = ax.boxplot(
                data,
                labels=labels,
                patch_artist=True,
                showfliers=show_fliers,
                medianprops=dict(color="black", linewidth=1.5)
            )

        # ---- style boxes: LP-only boxes in red, others by WagonType ----
        for box, wt, cfg in zip(bp["boxes"], box_wagon_types, box_configs):

            if cfg == "LP":
                box.set_facecolor("#d62728")   # red for LP
                box.set_alpha(0.85)
            else:
                box.set_facecolor(wagon_colors[wt])
                box.set_alpha(0.85)

            box.set_edgecolor("black")
            box.set_linewidth(1.2)

        if y_ref is not None:
            ax.axhline(y_ref, linestyle="--", color="black", linewidth=1.2)

        ax.set_title(f"{feat} — grouped by WagonType" + (" × Config" if separate_by_config else ""), fontsize=13)
        ax.set_xlabel(f"{source_col} groups")
        ax.set_ylabel(feat)
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_ylim(y_low, y_high)

        plt.xticks(rotation=45, ha="right")

        # ---- legend for Config (only when splitting) ----
        if separate_by_config:
            handles = [Patch(facecolor="white", edgecolor=config_colors[c], linewidth=2.0, label=f"{config_col} = {c}") for c in configs]
            ax.legend(handles=handles, loc="upper right", frameon=False)

        plt.tight_layout()
        plt.show()


In [ ]:
df_filter = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0)) &
    (df_monorail['First_phase_error'].eq(0))
].copy()

features_to_explore = [
    "First_phase_mean_curvature",
    "First_phase_half_time_ratio"
]

boxplot_features_by_source(
    df_filter, features_to_explore,
    source_col="WagonType",
    separate_by_config=True,
    ylim_mode="minmax",
    config_col="Config",
    configs=("HP", "LP")
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import chi2


def plot_mahalanobis_ellipse(mu, cov, T, ax, **kwargs):
    cov = 0.5 * (cov + cov.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    width, height = 2 * T * np.sqrt(eigvals)

    ell = Ellipse(xy=mu, width=width, height=height, angle=angle,
                  facecolor="none", **kwargs)
    ax.add_patch(ell)


def mahalanobis_group_metrics(X, alpha=0.95):
    mu = X.mean(axis=0)
    cov = np.cov(X, rowvar=False, ddof=1)
    cov = 0.5 * (cov + cov.T)

    # --- NEW: explicit covariance terms ---
    var_x = float(cov[0, 0])
    var_y = float(cov[1, 1])
    cov_xy = float(cov[0, 1])
    corr_xy = float(cov_xy / np.sqrt(var_x * var_y))  # optional but very useful
    theta = 0.5 * np.arctan2(2 * cov_xy, var_x - var_y)
    angle_deg = float(np.degrees(theta))

    
    eigvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
    lam1, lam2 = float(eigvals[0]), float(eigvals[1])

    chi2_thr = float(chi2.ppf(alpha, df=2))
    a = np.sqrt(chi2_thr * lam1)
    b = np.sqrt(chi2_thr * lam2)
    area = float(np.pi * a * b)

    cov_inv = np.linalg.inv(cov)
    d2 = np.sum((X - mu) @ cov_inv * (X - mu), axis=1)
    d = np.sqrt(d2)

    return {
        "n": int(X.shape[0]),
        "mu_x": float(mu[0]),
        "mu_y": float(mu[1]),

        # --- NEW: covariance info ---
        "var_x": var_x,
        "var_y": var_y,
        "cov_xy": cov_xy,
        "corr_xy": corr_xy,
        "ellipse_angle_deg": angle_deg,
        "det_cov": float(np.linalg.det(cov)),
        "eig1": lam1,
        "eig2": lam2,
        "anisotropy": float(lam1 / lam2),
        "ellipse_area": area,
        "mean_md": float(d.mean()),
        "md_p95": float(np.percentile(d, 95)),
        "outlier_rate": float((d2 > chi2_thr).mean())
    }



def plot_df_mahalanobis_by_wagontype(
    df,
    feature_cols,
    wagon_order=None,          # <-- YOU control order
    wagontype_col="WagonType",
    alpha=0.95,
    min_n=5,
    ax=None,
    figsize=(10, 10),
    xlim=None,
    ylim=None,
    return_metrics=False
):
    d = df[[*feature_cols, wagontype_col]].dropna().copy()
    d[feature_cols] = d[feature_cols].apply(pd.to_numeric, errors="coerce")
    d = d.dropna(subset=feature_cols)

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    colors = [
        "#0173B2", "#CC78BC", "#03FF74", "#2C5376",
        "#CA9161", "#949494", "#ECE133", "#56B4E9",
    ]

    T = np.sqrt(chi2.ppf(alpha, df=2))

    if wagon_order is None:
        wagon_types = list(d[wagontype_col].unique())
    else:
        wagon_types = list(wagon_order)

    metrics_rows = []

    for idx, wt in enumerate(wagon_types):
        g = d[d[wagontype_col] == wt]
        if len(g) < min_n:
            continue

        Xg = g[feature_cols].to_numpy(float)
        color = colors[idx % len(colors)]

        mu = Xg.mean(axis=0)
        cov = np.cov(Xg, rowvar=False, ddof=1)

        # scatter + centroid + ellipse
        ax.scatter(Xg[:, 0], Xg[:, 1], s=30, alpha=0.5, color=color, label=str(wt), edgecolors="none")
        ax.scatter(mu[0], mu[1], marker="^", s=180, facecolor=color, edgecolor="black", linewidth=1.2, zorder=5)
        plot_mahalanobis_ellipse(mu, cov, T, ax=ax, edgecolor=color, linewidth=2.5, linestyle="--", alpha=0.4)

        m = mahalanobis_group_metrics(Xg, alpha=alpha)
        m[wagontype_col] = str(wt)
        metrics_rows.append(m)

    ax.set_title(f"Mahalanobis Distance Ellipses by {wagontype_col}", fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel(feature_cols[0], fontsize=12, fontweight="bold")
    ax.set_ylabel(feature_cols[1], fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3, linestyle="--", linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_aspect("equal", adjustable="box")

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    leg = ax.legend(title=wagontype_col, title_fontsize=11, fontsize=10, loc="best",
                    framealpha=0.95, edgecolor="black", fancybox=False)
    leg.get_title().set_fontweight("bold")

    plt.tight_layout()

    metrics_df = pd.DataFrame(metrics_rows).set_index(wagontype_col)

    if return_metrics:
        return ax, metrics_df
    return ax


In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = ["Total_power_efficiency", "Std_delay_exp"]

scaler = StandardScaler()

df_scaled = df_bchealthy.copy()
df_scaled[feature_cols] = scaler.fit_transform(df_bchealthy[feature_cols])


In [ ]:
df_compare = df_bchealthy[df_bchealthy['WV_MeanPressureLevel'] == 1].copy()

In [ ]:
features_ci = ["Total_power_efficiency", "Std_delay_exp"]

# Safety checks
missing = [f for f in features_ci if f not in df_bchealthy.columns]
if missing:
    raise ValueError(f"Missing features in df_bchealthy: {missing}")

# Compute non-parametric 95% CI (2.5% – 97.5%)
ci_bounds = {}

for f in features_ci:
    lower = df_bchealthy[f].quantile(0.05)
    upper = df_bchealthy[f].quantile(0.95)
    ci_bounds[f] = (lower, upper)

ci_bounds
mask_ci = np.ones(len(df_bchealthy), dtype=bool)

for f, (low, high) in ci_bounds.items():
    mask_ci &= df_bchealthy[f].between(low, high)

df_bchealthy_ci = df_bchealthy.loc[mask_ci].copy()

print(f"Original healthy samples: {len(df_bchealthy)}")
print(f"After 95% CI filtering:   {len(df_bchealthy_ci)}")
print(f"Removed as outliers:      {len(df_bchealthy) - len(df_bchealthy_ci)}")


In [ ]:
for f, (low, high) in ci_bounds.items():
    out = ~df_bchealthy[f].between(low, high)
    print(
        f"{f}: "
        f"{out.sum()} / {len(df_bchealthy)} "
        f"({100*out.mean():.2f}%) outside 95% CI"
    )


In [ ]:
# ------------------ Example usage ------------------

# Choose the TWO features you want on x/y axes
feature_cols = ["Total_power_efficiency", "Std_delay_exp"]

counts = df_bchealthy_ci["WagonType"].value_counts()
wagon_order = counts.sort_values(ascending=False).index.tolist()  # smallest first

ax, metrics_df = plot_df_mahalanobis_by_wagontype(
    df_bchealthy_ci,
    feature_cols=["Total_power_efficiency", "Std_delay_exp"],
    wagon_order=wagon_order,
    return_metrics=True
)

# print(metrics_df[["n","mu_x","mu_y" ,"det_cov", "anisotropy", "ellipse_area", "mean_md", "md_p95", "outlier_rate"]])

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

def boxplot_features_by_wagontype_config_plotly(
    df,
    features,
    wagon_col="WagonType",
    config_col="Config",          # <-- NEW
    wagon_filter=None,
    y_ref=None,
    ylim_mode="robust",
    margin=0.10,
    show_fliers=True,
    # ---- layout controls ----
    xlabel="WagonType | Config",
    ylabel=None,
    title_fmt="{feat} — of Wagon Type | Sensor Config",
    x_order=None,                 # explicit order recommended
    height=500,
    width=None,
):
    """
    Plotly interactive boxplots grouped by WagonType | Config.

    Example X labels:
        T3000 | HP
        4909  | HP
        4575  | HP
        T3000 | LP
    """

    df = df.copy()

    if wagon_filter is not None:
        df = df[df[wagon_col].isin([wagon_filter])]

    if df.empty:
        print("[SKIP] Empty dataframe after filtering.")
        return

    # ensure categorical strings
    df[wagon_col]  = df[wagon_col].astype(str)
    df[config_col] = df[config_col].astype(str)

    # ---- build composite x label ----
    df["_WagonConfig"] = df[wagon_col] + " | " + df[config_col]

    for feat in features:
        if feat not in df.columns:
            print(f"[SKIP] Feature not found: {feat}")
            continue

        tmp = df[["_WagonConfig", wagon_col, feat]].copy()
        tmp[feat] = pd.to_numeric(tmp[feat], errors="coerce")
        tmp = tmp.dropna(subset=[feat])

        if tmp.empty:
            print(f"[SKIP] No valid data for feature: {feat}")
            continue

        vals = tmp[feat].to_numpy()

        # ---- y-limits ----
        if ylim_mode == "robust":
            y_low  = float(np.nanpercentile(vals, 1))
            y_high = float(np.nanpercentile(vals, 99))
        else:
            y_low  = min(0.0, float(np.nanmin(vals)))
            y_high = float(np.nanmax(vals))

        if y_high <= y_low:
            y_high = y_low + 1.0
        y_high *= (1 + margin)

        points = "outliers" if show_fliers else False

        fig = px.box(
            tmp,
            x="_WagonConfig",
            y=feat,
            color=wagon_col,
            points=points,
            title=title_fmt.format(feat=feat),  # <-- string only
            category_orders={"_WagonConfig": x_order} if x_order is not None else None,
        )

        fig.update_layout(
            title=dict(
                text=title_fmt.format(feat=feat),
                x=0.5,
                xanchor="center",
                yanchor="top",
                font=dict(size=26, family="Arial", color="#333333")
            ),
            xaxis=dict(
                title=dict(text=xlabel, font=dict(size=22, family="Arial", color="#333333")),
                tickfont=dict(size=20, family="Arial", color="#333333")
            ),
            yaxis=dict(
                title=dict(text=(feat if ylabel is None else ylabel), font=dict(size=22, family="Arial", color="#333333")),
                tickfont=dict(size=20, family="Arial", color="#333333")
            ),
            height=height,
            width=width,
            margin=dict(l=40, r=20, t=60, b=100),
            legend_title_text="WagonType",
        )
        fig.update_xaxes(
            categoryorder="array",
            categoryarray=x_order
        )

        fig.update_traces(
            width=0.6
        )

        fig.update_xaxes(tickangle=0)
        fig.update_yaxes(range=[y_low, y_high])

        if y_ref is not None:
            fig.add_hline(y=y_ref, line_dash="dash", line_width=2)

        fig.show()


# Compare Wagon

In [ ]:
df_filter = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0))
].copy()

features_ci = ["Total_power_efficiency", "Std_delay_exp"]

# Safety checks
missing = [f for f in features_ci if f not in df_filter.columns]
if missing:
    raise ValueError(f"Missing features in df_filter: {missing}")

# Compute non-parametric 95% CI (2.5% – 97.5%)
ci_bounds = {}

for f in features_ci:
    lower = df_filter[f].quantile(0.05)
    upper = df_filter[f].quantile(0.95)
    ci_bounds[f] = (lower, upper)

ci_bounds
mask_ci = np.ones(len(df_filter), dtype=bool)

for f, (low, high) in ci_bounds.items():
    mask_ci &= df_filter[f].between(low, high)

df_filter_ci = df_filter.loc[mask_ci].copy()

print(f"Original samples: {len(df_filter)}")
print(f"After 95% CI filtering:   {len(df_filter_ci)}")
print(f"Removed as outliers:      {len(df_filter) - len(df_filter_ci)}")

x_order = [
    "T3000 | HP",
    "T3000 | LP",   

]

boxplot_features_by_wagontype_config_plotly(
    df=df_filter_ci,
    features=[
        "Total_power_efficiency",
        "Std_delay_exp",
    ],
    x_order=x_order,
    y_ref=None,
    show_fliers=True
)

In [ ]:
df_filter = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0))
].copy()

features_ci = ["Total_power_efficiency", "Std_delay_exp"]

# Safety checks
missing = [f for f in features_ci if f not in df_filter.columns]
if missing:
    raise ValueError(f"Missing features in df_filter: {missing}")

# Compute non-parametric 95% CI (2.5% – 97.5%)
ci_bounds = {}

for f in features_ci:
    lower = df_filter[f].quantile(0.05)
    upper = df_filter[f].quantile(0.95)
    ci_bounds[f] = (lower, upper)

ci_bounds
mask_ci = np.ones(len(df_filter), dtype=bool)

for f, (low, high) in ci_bounds.items():
    mask_ci &= df_filter[f].between(low, high)

df_filter_ci = df_filter.loc[mask_ci].copy()

print(f"Original samples: {len(df_filter)}")
print(f"After 95% CI filtering:   {len(df_filter_ci)}")
print(f"Removed as outliers:      {len(df_filter) - len(df_filter_ci)}")

x_order = [
    "T3000 | HP",  
    "4575 | HP"
]

boxplot_features_by_wagontype_config_plotly(
    df=df_filter_ci,
    features=[
        "Total_power_efficiency",
        "Std_delay_exp",
    ],
    x_order=x_order,
    y_ref=None,
    show_fliers=True
)

In [ ]:
df_filter = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0))
].copy()

features_ci = ["Total_power_efficiency", "Std_delay_exp"]

# Safety checks
missing = [f for f in features_ci if f not in df_filter.columns]
if missing:
    raise ValueError(f"Missing features in df_filter: {missing}")

# Compute non-parametric 95% CI (2.5% – 97.5%)
ci_bounds = {}

for f in features_ci:
    lower = df_filter[f].quantile(0.05)
    upper = df_filter[f].quantile(0.95)
    ci_bounds[f] = (lower, upper)

ci_bounds
mask_ci = np.ones(len(df_filter), dtype=bool)

for f, (low, high) in ci_bounds.items():
    mask_ci &= df_filter[f].between(low, high)

df_filter_ci = df_filter.loc[mask_ci].copy()

print(f"Original samples: {len(df_filter)}")
print(f"After 95% CI filtering:   {len(df_filter_ci)}")
print(f"Removed as outliers:      {len(df_filter) - len(df_filter_ci)}")

x_order = [
    "T3000 | HP",
    "4909 | HP", 
    "4575 | HP"
]

boxplot_features_by_wagontype_config_plotly(
    df=df_filter_ci,
    features=[
        "Total_power_efficiency",
        "Std_delay_exp",
    ],
    x_order=x_order,
    y_ref=None,
    show_fliers=True
)

In [ ]:
df_filter = df_monorail.loc[
    (df_monorail['Non_Standard_Braking'].eq(0)) &
    (df_monorail['BC_BadStart'].eq(0))
].copy()

features_ci = ["First_phase_mean_curvature"]

# Safety checks
missing = [f for f in features_ci if f not in df_filter.columns]
if missing:
    raise ValueError(f"Missing features in df_filter: {missing}")

# Compute non-parametric 95% CI (2.5% – 97.5%)
ci_bounds = {}

for f in features_ci:
    lower = df_filter[f].quantile(0.025)
    upper = df_filter[f].quantile(0.975)
    ci_bounds[f] = (lower, upper)

ci_bounds
mask_ci = np.ones(len(df_filter), dtype=bool)

for f, (low, high) in ci_bounds.items():
    mask_ci &= df_filter[f].between(low, high)

df_filter_ci = df_filter.loc[mask_ci].copy()

print(f"Original samples: {len(df_filter)}")
print(f"After 95% CI filtering:   {len(df_filter_ci)}")
print(f"Removed as outliers:      {len(df_filter) - len(df_filter_ci)}")

x_order = [
    "T3000 | HP",
    "T3000 | LP",   
    "4909 | HP",
    "4575 | HP",
]

boxplot_features_by_wagontype_config_plotly(
    df=df_filter_ci,
    features=[
        "First_phase_mean_curvature",
        "First_phase_half_time_ratio",
        "First_phase_power",
    ],
    x_order=x_order,
    y_ref=None,
    show_fliers=True
)


In [ ]:
import numpy as np
import pandas as pd

def mean_median_diff_vs_reference(
    df,
    features,
    group_col="_WagonConfig",
    ref_group="T3000 | HP"
):
    """
    Compute mean and median differences per Wagon|Config
    relative to a reference group.

    Returns a tidy DataFrame with:
    - Feature
    - Group
    - N
    - Mean
    - Median
    - Mean_diff_vs_ref
    - Median_diff_vs_ref
    - Mean_diff_pct_vs_ref
    - Median_diff_pct_vs_ref
    """

    rows = []

    for feat in features:
        tmp = df[[group_col, feat]].dropna()

        if ref_group not in tmp[group_col].unique():
            raise ValueError(
                f"Reference group '{ref_group}' not found for feature '{feat}'"
            )

        ref_vals = tmp.loc[tmp[group_col] == ref_group, feat]
        ref_mean   = ref_vals.mean()
        ref_median = ref_vals.median()

        for grp, gdf in tmp.groupby(group_col):
            mean_g   = gdf[feat].mean()
            median_g = gdf[feat].median()

            rows.append({
                "Feature": feat,
                "Group": grp,
                "N": len(gdf),
                "Mean": mean_g,
                "Median": median_g,
                "Mean_diff_vs_T3000_HP": mean_g - ref_mean,
                "Median_diff_vs_T3000_HP": median_g - ref_median,
                "Mean_diff_pct_vs_T3000_HP": (
                    100 * (mean_g - ref_mean) / ref_mean if ref_mean != 0 else np.nan
                ),
                "Median_diff_pct_vs_T3000_HP": (
                    100 * (median_g - ref_median) / ref_median if ref_median != 0 else np.nan
                ),
            })

    out = (
        pd.DataFrame(rows)
        .sort_values(["Feature", "Group"])
        .reset_index(drop=True)
    )

    return out


In [ ]:
df_filter_ci["_WagonConfig"] = df_filter_ci["WagonType"] + " | " + df_filter_ci["Config"]

var_table = mean_median_diff_vs_reference(
    df=df_filter_ci,
    features=[
        "Total_power_efficiency",
        "Std_delay_exp",
    ],
    ref_group="T3000 | HP"
)
var_table.to_excel("mean_comparison_table.xlsx", index=False)
var_table


In [ ]:
import numpy as np

def centroid_distance(X_ref, X):
    mu_ref = X_ref.mean(axis=0)
    mu = X.mean(axis=0)
    return np.linalg.norm(mu - mu_ref)


In [ ]:
metrics_df

In [ ]:
features = ["Total_power_efficiency", "Std_delay_exp"]

df_regime = df_bchealthy_ci[
    (df_bchealthy_ci["WV_MeanPressureLevel"] == 1) &               # example: WV 2–3 bar
    (df_bchealthy_ci["Non_Standard_Braking"] == 0) &
    (df_bchealthy_ci["BC_BadStart"] == 0)
]
df_T3000 = df_regime[df_regime["WagonType"] == "T3000"]
df_4575  = df_regime[df_regime["WagonType"] == "4575"]
df_4909  = df_regime[df_regime["WagonType"] == "4909"]
X_ref = df_T3000[features].to_numpy()
X_4575 = df_4575[features].to_numpy()
X_4909 = df_4909[features].to_numpy()

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

scaler = StandardScaler().fit(X_ref)
imputer = SimpleImputer(strategy="median")

X_ref = scaler.transform(X_ref)
X_4575 = scaler.transform(X_4575)
X_4909 = scaler.transform(X_4909)

X_ref = imputer.fit_transform(X_ref)
X_4575 = imputer.transform(X_4575)
X_4909 = imputer.transform(X_4909)


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import numpy as np

def principal_angle(X_ref, X):
    pca_ref = PCA(n_components=1).fit(X_ref)
    pca = PCA(n_components=1).fit(X)

    v_ref = pca_ref.components_[0]
    v = pca.components_[0]

    cos_theta = np.abs(np.dot(v_ref, v))
    theta = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
    return theta
def centroid_distance(X_ref, X):
    mu_ref = X_ref.mean(axis=0)
    mu = X.mean(axis=0)
    return np.linalg.norm(mu - mu_ref)

def variance_ratio(X_ref, X):
    lam_ref = PCA(n_components=1).fit(X_ref).explained_variance_[0]
    lam = PCA(n_components=1).fit(X).explained_variance_[0]
    return lam / lam_ref
from scipy.linalg import inv

def mahalanobis_centroid(X_ref, X):
    mu_ref = X_ref.mean(axis=0)
    mu = X.mean(axis=0)
    cov_ref = np.cov(X_ref, rowvar=False)
    diff = mu - mu_ref
    return np.sqrt(diff @ inv(cov_ref) @ diff)


In [ ]:
results = {
    "4575": {
        "centroid_dist": centroid_distance(X_ref, X_4575),
        "pca_angle_deg": principal_angle(X_ref, X_4575),
        "var_ratio": variance_ratio(X_ref, X_4575),
        "mahalanobis": mahalanobis_centroid(X_ref, X_4575),
    },
    "4909": {
        "centroid_dist": centroid_distance(X_ref, X_4909),
        "pca_angle_deg": principal_angle(X_ref, X_4909),
        "var_ratio": variance_ratio(X_ref, X_4909),
        "mahalanobis": mahalanobis_centroid(X_ref, X_4909),
    }
}

results

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def _angle_deg_2d(vec):
    """Absolute orientation angle in degrees for a 2D vector."""
    return float(np.degrees(np.arctan2(vec[1], vec[0])))

def _angle_between_deg(u, v):
    """Angle between vectors in degrees (dimension-agnostic)."""
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    denom = (np.linalg.norm(u) * np.linalg.norm(v))
    if denom == 0:
        return np.nan
    cos = np.abs(np.dot(u, v) / denom)  # abs -> sign-invariant eigenvectors
    return float(np.degrees(np.arccos(np.clip(cos, -1, 1))))

def build_centroid_pca_table(
    df: pd.DataFrame,
    features: list[str],
    wagon_col: str = "WagonType",
    ref_wagon: str = "T3000",
    regime_col: str | None = "WV_bin",
    regimes: list | None = None,
    min_samples: int = 10
) -> pd.DataFrame:
    """
    Returns a table with (per regime, per wagon):
    - N
    - raw centroid components (mean of features, BEFORE scaling)
    - scaled centroid components (AFTER scaling with ref scaler)
    - PC1 loadings (in scaled space)
    - PC1 absolute angle (deg) if d == 2
    - PC1 angle relative to reference PC1 (deg)
    """

    # checks
    required = [wagon_col] + features
    if regime_col is not None:
        required += [regime_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    work = df.copy()

    # regime handling
    if regime_col is None:
        work["_REGIME_"] = "ALL"
        regime_use = "_REGIME_"
    else:
        regime_use = regime_col

    if regimes is None:
        regimes = sorted(work[regime_use].dropna().unique().tolist())

    rows = []
    d = len(features)

    for reg in regimes:
        df_r = work[work[regime_use] == reg]

        # Reference wagon in this regime
        df_ref = df_r[df_r[wagon_col] == ref_wagon]
        if len(df_ref) < min_samples:
            continue

        X_ref_raw = df_ref[features].to_numpy(dtype=float)

        # fit scaler on reference only (per regime)
        scaler = StandardScaler().fit(X_ref_raw)
        X_ref_scaled = scaler.transform(X_ref_raw)

        # reference PCA direction (PC1) in scaled space
        pca_ref = PCA(n_components=1).fit(X_ref_scaled)
        v_ref = pca_ref.components_[0]

        wagons_here = sorted(df_r[wagon_col].dropna().unique().tolist())
        if ref_wagon not in wagons_here:
            wagons_here = [ref_wagon] + wagons_here

        for w in wagons_here:
            df_w = df_r[df_r[wagon_col] == w]
            if len(df_w) < min_samples:
                continue

            X_raw = df_w[features].to_numpy(dtype=float)
            X_scaled = scaler.transform(X_raw)

            # raw centroid (original units)
            mu_raw = np.nanmean(X_raw, axis=0)

            # scaled centroid (comparable across wagons within regime)
            mu_scaled = np.nanmean(X_scaled, axis=0)

            # PCA PC1 for this wagon (scaled space)
            pca = PCA(n_components=1).fit(X_scaled)
            v = pca.components_[0]
            lam1 = float(pca.explained_variance_[0])
            evr1 = float(pca.explained_variance_ratio_[0])

            row = {
                "Regime": reg,
                "WagonType": w,
                "N": len(df_w),
                "PC1_lambda": lam1,
                "PC1_EVR": evr1,
                "PC1_angle_to_ref_deg": _angle_between_deg(v_ref, v),
                "ref_wagon": ref_wagon,
            }

            # Add centroid components
            for j, feat in enumerate(features):
                row[f"mu_raw_{feat}"] = float(mu_raw[j])
                row[f"mu_scaled_{feat}"] = float(mu_scaled[j])

            # Add PC1 loadings
            for j, feat in enumerate(features):
                row[f"pc1_loading_{feat}"] = float(v[j])

            # Add absolute PC1 orientation only when 2D
            if d == 2:
                row["PC1_abs_angle_deg"] = _angle_deg_2d(v)
                row["Centroid_abs_angle_deg"] = _angle_deg_2d(mu_scaled)  # optional

            rows.append(row)

    out = pd.DataFrame(rows)

    # nice ordering: baseline first in each regime
    if not out.empty:
        out["is_ref"] = (out["WagonType"] == ref_wagon).astype(int)
        out = out.sort_values(["Regime", "is_ref", "PC1_angle_to_ref_deg"],
                              ascending=[True, False, True]).drop(columns="is_ref")

    return out


In [ ]:
features = ["Total_power_efficiency", "Std_delay_exp"]  # example
table = build_centroid_pca_table(
    df=df_regime,
    features=features,
    wagon_col="WagonType",
    ref_wagon="T3000",
    regime_col="WV_MeanPressureLevel",
    min_samples=10
)
table.to_excel("statistical_data.xlsx", index=False)

print(table)


In [ ]:
from numpy.linalg import inv

baseline = "T3000"

# Recompute reference covariance from raw data
X_ref = df_compare.loc[
    df_compare["WagonType"] == baseline,
    feature_cols
].dropna().values

cov_ref = np.cov(X_ref, rowvar=False, ddof=1)
cov_ref += 1e-9 * np.trace(cov_ref) * np.eye(2)
cov_ref_inv = inv(cov_ref)

mu_ref = metrics_df.loc[baseline, ["mu_x", "mu_y"]].values

def centroid_mahalanobis(mu):
    d = mu - mu_ref
    return float(np.sqrt(d @ cov_ref_inv @ d))

metrics_df["centroid_MD_to_T3000"] = (
    metrics_df[["mu_x", "mu_y"]]
    .apply(lambda r: centroid_mahalanobis(r.values), axis=1)
)

ref_disp = np.sqrt(metrics_df.loc[baseline, "det_cov"])

metrics_df["dispersion_var_pct"] = (
    (np.sqrt(metrics_df["det_cov"]) - ref_disp) / ref_disp
) * 100

metrics_df["ellipse_area_var_pct"] = (
    (metrics_df["ellipse_area"] -
     metrics_df.loc[baseline, "ellipse_area"])
    / metrics_df.loc[baseline, "ellipse_area"]
) * 100

metrics_df["var_major_pct"] = (
    (np.sqrt(metrics_df["eig1"]) -
     np.sqrt(metrics_df.loc[baseline, "eig1"]))
    / np.sqrt(metrics_df.loc[baseline, "eig1"])
) * 100

metrics_df["var_minor_pct"] = (
    (np.sqrt(metrics_df["eig2"]) -
     np.sqrt(metrics_df.loc[baseline, "eig2"]))
    / np.sqrt(metrics_df.loc[baseline, "eig2"])
) * 100
metrics_df["compactness_var_pct"] = (
    (metrics_df["mean_md"] -
     metrics_df.loc[baseline, "mean_md"])
    / metrics_df.loc[baseline, "mean_md"]
) * 100

metrics_df


In [ ]:
# ------------------ Example usage ------------------
# Choose the TWO features you want on x/y axes
feature_cols = ["Total_power_efficiency", "Std_delay_exp"]

ax, metrics_df = plot_df_mahalanobis_by_wagontype(
    df_compare[df_compare["WagonType"].isin(["T3000","4909"])],
    feature_cols=feature_cols,
    wagontype_col="WagonType",
    alpha=0.95,
    # xlim=(0, 9),
    figsize=(10, 8),
    return_metrics=True
)
print(metrics_df[["n","mu_x","mu_y" ,"det_cov", "anisotropy", "ellipse_area", "mean_md", "md_p95", "outlier_rate"]])

plt.show()

In [ ]:
from numpy.linalg import inv

baseline = "T3000"

# Recompute reference covariance from raw data
X_ref = df_compare.loc[
    df_compare["WagonType"] == baseline,
    feature_cols
].dropna().values

cov_ref = np.cov(X_ref, rowvar=False, ddof=1)
cov_ref += 1e-9 * np.trace(cov_ref) * np.eye(2)
cov_ref_inv = inv(cov_ref)

mu_ref = metrics_df.loc[baseline, ["mu_x", "mu_y"]].values

def centroid_mahalanobis(mu):
    d = mu - mu_ref
    return float(np.sqrt(d @ cov_ref_inv @ d))

metrics_df["centroid_MD_to_T3000"] = (
    metrics_df[["mu_x", "mu_y"]]
    .apply(lambda r: centroid_mahalanobis(r.values), axis=1)
)

ref_disp = np.sqrt(metrics_df.loc[baseline, "det_cov"])

metrics_df["dispersion_var_pct"] = (
    (np.sqrt(metrics_df["det_cov"]) - ref_disp) / ref_disp
) * 100

metrics_df["ellipse_area_var_pct"] = (
    (metrics_df["ellipse_area"] -
     metrics_df.loc[baseline, "ellipse_area"])
    / metrics_df.loc[baseline, "ellipse_area"]
) * 100

metrics_df["var_major_pct"] = (
    (np.sqrt(metrics_df["eig1"]) -
     np.sqrt(metrics_df.loc[baseline, "eig1"]))
    / np.sqrt(metrics_df.loc[baseline, "eig1"])
) * 100

metrics_df["var_minor_pct"] = (
    (np.sqrt(metrics_df["eig2"]) -
     np.sqrt(metrics_df.loc[baseline, "eig2"]))
    / np.sqrt(metrics_df.loc[baseline, "eig2"])
) * 100
metrics_df["compactness_var_pct"] = (
    (metrics_df["mean_md"] -
     metrics_df.loc[baseline, "mean_md"])
    / metrics_df.loc[baseline, "mean_md"]
) * 100

metrics_df


In [ ]:
# ------------------ Example usage ------------------
# Choose the TWO features you want on x/y axes
feature_cols = ["Total_power_efficiency", "Std_delay_exp"]

ax, metrics_df = plot_df_mahalanobis_by_wagontype(
    df_scaled,
    feature_cols=feature_cols,
    wagontype_col="WagonType",
    alpha=0.95,
    figsize=(10, 8),
    return_metrics=True
)
print(metrics_df[["n","mu_x","mu_y" ,"det_cov", "anisotropy", "ellipse_area", "mean_md", "md_p95", "outlier_rate"]])

plt.show()

In [ ]:
from numpy.linalg import inv

baseline = "T3000"

# Recompute reference covariance from raw data
X_ref = df_bchealthy.loc[
    df_bchealthy["WagonType"] == baseline,
    feature_cols
].dropna().values

cov_ref = np.cov(X_ref, rowvar=False, ddof=1)
cov_ref += 1e-9 * np.trace(cov_ref) * np.eye(2)
cov_ref_inv = inv(cov_ref)

mu_ref = metrics_df.loc[baseline, ["mu_x", "mu_y"]].values

def centroid_mahalanobis(mu):
    d = mu - mu_ref
    return float(np.sqrt(d @ cov_ref_inv @ d))

metrics_df["centroid_MD_to_T3000"] = (
    metrics_df[["mu_x", "mu_y"]]
    .apply(lambda r: centroid_mahalanobis(r.values), axis=1)
)

ref_disp = np.sqrt(metrics_df.loc[baseline, "det_cov"])

metrics_df["dispersion_var_pct"] = (
    (np.sqrt(metrics_df["det_cov"]) - ref_disp) / ref_disp
) * 100

metrics_df["ellipse_area_var_pct"] = (
    (metrics_df["ellipse_area"] -
     metrics_df.loc[baseline, "ellipse_area"])
    / metrics_df.loc[baseline, "ellipse_area"]
) * 100

metrics_df["var_major_pct"] = (
    (np.sqrt(metrics_df["eig1"]) -
     np.sqrt(metrics_df.loc[baseline, "eig1"]))
    / np.sqrt(metrics_df.loc[baseline, "eig1"])
) * 100

metrics_df["var_minor_pct"] = (
    (np.sqrt(metrics_df["eig2"]) -
     np.sqrt(metrics_df.loc[baseline, "eig2"]))
    / np.sqrt(metrics_df.loc[baseline, "eig2"])
) * 100
metrics_df["compactness_var_pct"] = (
    (metrics_df["mean_md"] -
     metrics_df.loc[baseline, "mean_md"])
    / metrics_df.loc[baseline, "mean_md"]
) * 100

metrics_df


In [ ]:
# ------------------ Example usage ------------------


# Choose the TWO features you want on x/y axes
feature_cols = ["Total_power_delay", "Std_delay_exp"]  # <-- replace with your actual columns

ax, metrics_df = plot_df_mahalanobis_by_wagontype(
    df_bchealthy,
    feature_cols=feature_cols,
    wagontype_col="WagonType",
    alpha=0.95,
    figsize=(10, 8),
    return_metrics=True
)
print(metrics_df[["n","mu_x","mu_y" ,"det_cov", "anisotropy", "ellipse_area", "mean_md", "md_p95", "outlier_rate"]])

plt.show()

In [ ]:
from numpy.linalg import inv

baseline = "T3000"

# Recompute reference covariance from raw data
X_ref = df_bchealthy.loc[
    df_bchealthy["WagonType"] == baseline,
    feature_cols
].dropna().values

cov_ref = np.cov(X_ref, rowvar=False, ddof=1)
cov_ref += 1e-9 * np.trace(cov_ref) * np.eye(2)
cov_ref_inv = inv(cov_ref)

mu_ref = metrics_df.loc[baseline, ["mu_x", "mu_y"]].values

def centroid_mahalanobis(mu):
    d = mu - mu_ref
    return float(np.sqrt(d @ cov_ref_inv @ d))

metrics_df["centroid_MD_to_T3000"] = (
    metrics_df[["mu_x", "mu_y"]]
    .apply(lambda r: centroid_mahalanobis(r.values), axis=1)
)

ref_disp = np.sqrt(metrics_df.loc[baseline, "det_cov"])

metrics_df["dispersion_var_pct"] = (
    (np.sqrt(metrics_df["det_cov"]) - ref_disp) / ref_disp
) * 100

metrics_df["ellipse_area_var_pct"] = (
    (metrics_df["ellipse_area"] -
     metrics_df.loc[baseline, "ellipse_area"])
    / metrics_df.loc[baseline, "ellipse_area"]
) * 100

metrics_df["var_major_pct"] = (
    (np.sqrt(metrics_df["eig1"]) -
     np.sqrt(metrics_df.loc[baseline, "eig1"]))
    / np.sqrt(metrics_df.loc[baseline, "eig1"])
) * 100

metrics_df["var_minor_pct"] = (
    (np.sqrt(metrics_df["eig2"]) -
     np.sqrt(metrics_df.loc[baseline, "eig2"]))
    / np.sqrt(metrics_df.loc[baseline, "eig2"])
) * 100
metrics_df["compactness_var_pct"] = (
    (metrics_df["mean_md"] -
     metrics_df.loc[baseline, "mean_md"])
    / metrics_df.loc[baseline, "mean_md"]
) * 100

metrics_df


In [ ]:
# ------------------ Example usage ------------------
# Choose the TWO features you want on x/y axes
feature_cols = ["Total_power_efficiency", "Std_delay_exp"]  # <-- replace with your actual columns

ax, metrics_df = plot_df_mahalanobis_by_wagontype(
    df_bchealthy[df_bchealthy["WV_MeanPressureLevel"] == 1],
    feature_cols=feature_cols,
    wagontype_col="WagonType",
    alpha=0.95,
    return_metrics=True
)
print(metrics_df[["n","mu_x","mu_y" ,"det_cov", "anisotropy", "ellipse_area", "mean_md", "md_p95", "outlier_rate"]])

plt.show()

In [ ]:
from numpy.linalg import inv

baseline = "T3000"

# Recompute reference covariance from raw data
X_ref = df_bchealthy.loc[
    df_bchealthy["WagonType"] == baseline,
    feature_cols
].dropna().values

cov_ref = np.cov(X_ref, rowvar=False, ddof=1)
cov_ref += 1e-9 * np.trace(cov_ref) * np.eye(2)
cov_ref_inv = inv(cov_ref)

mu_ref = metrics_df.loc[baseline, ["mu_x", "mu_y"]].values

def centroid_mahalanobis(mu):
    d = mu - mu_ref
    return float(np.sqrt(d @ cov_ref_inv @ d))

metrics_df["centroid_MD_to_T3000"] = (
    metrics_df[["mu_x", "mu_y"]]
    .apply(lambda r: centroid_mahalanobis(r.values), axis=1)
)

ref_disp = np.sqrt(metrics_df.loc[baseline, "det_cov"])

metrics_df["dispersion_var_pct"] = (
    (np.sqrt(metrics_df["det_cov"]) - ref_disp) / ref_disp
) * 100

metrics_df["ellipse_area_var_pct"] = (
    (metrics_df["ellipse_area"] -
     metrics_df.loc[baseline, "ellipse_area"])
    / metrics_df.loc[baseline, "ellipse_area"]
) * 100

metrics_df["var_major_pct"] = (
    (np.sqrt(metrics_df["eig1"]) -
     np.sqrt(metrics_df.loc[baseline, "eig1"]))
    / np.sqrt(metrics_df.loc[baseline, "eig1"])
) * 100

metrics_df["var_minor_pct"] = (
    (np.sqrt(metrics_df["eig2"]) -
     np.sqrt(metrics_df.loc[baseline, "eig2"]))
    / np.sqrt(metrics_df.loc[baseline, "eig2"])
) * 100
metrics_df["compactness_var_pct"] = (
    (metrics_df["mean_md"] -
     metrics_df.loc[baseline, "mean_md"])
    / metrics_df.loc[baseline, "mean_md"]
) * 100

metrics_df


# PCA - SCREE PLOT

In [ ]:
selected_features = [
    "Total_power_delay",
    "Total_power_efficiency",
    "Std_delay_exp",
    "Total_energy_efficiency",
    "Total_energy_delay"
    
]

df_T3000.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pandas as pd

# Standardize the features
scaler = StandardScaler()
imputer = SimpleImputer(strategy='median')
X_scaled = scaler.fit_transform(df_T3000[selected_features])
X_scaled = imputer.fit_transform(X_scaled)
# Apply PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Create a figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Scree Plot - Explained Variance
axes[0, 0].bar(range(1, len(pca.explained_variance_ratio_) + 1), 
               pca.explained_variance_ratio_, 
               alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Principal Component', fontsize=12)
axes[0, 0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0, 0].set_title('Scree Plot - Variance Explained by Each PC', fontsize=14, fontweight='bold')
axes[0, 0].set_xticks(range(1, len(pca.explained_variance_ratio_) + 1))

# 2. Cumulative Explained Variance
cumsum = np.cumsum(pca.explained_variance_ratio_)
axes[0, 1].plot(range(1, len(cumsum) + 1), cumsum, marker='o', linestyle='-', color='darkorange', linewidth=2)
axes[0, 1].axhline(y=0.95, color='r', linestyle='--', label='95% Variance')
axes[0, 1].set_xlabel('Number of Components', fontsize=12)
axes[0, 1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[0, 1].set_title('Cumulative Explained Variance', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)
axes[0, 1].set_xticks(range(1, len(cumsum) + 1))

# 3. PCA Biplot (PC1 vs PC2)
axes[1, 0].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=30, color='gray')
axes[1, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
axes[1, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
axes[1, 0].set_title('PCA Scatter Plot (PC1 vs PC2)', fontsize=14, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Add loading vectors
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
for i, feature in enumerate(selected_features):
    axes[1, 0].arrow(0, 0, loadings[i, 0]*3, loadings[i, 1]*3, 
                     head_width=0.1, head_length=0.1, fc='red', ec='red', alpha=0.7)
    axes[1, 0].text(loadings[i, 0]*3.2, loadings[i, 1]*3.2, feature, 
                    fontsize=9, ha='center', color='darkred', fontweight='bold')

# 4. Feature Loadings Heatmap for PC1 and PC2
loadings_df = pd.DataFrame(
    pca.components_[:2, :].T,
    columns=['PC1', 'PC2'],
    index=selected_features
)

im = axes[1, 1].imshow(loadings_df.T, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[1, 1].set_xticks(range(len(selected_features)))
axes[1, 1].set_xticklabels(selected_features, rotation=45, ha='right')
axes[1, 1].set_yticks(range(2))
axes[1, 1].set_yticklabels(['PC1', 'PC2'])
axes[1, 1].set_title('Feature Loadings Heatmap', fontsize=14, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=axes[1, 1])
cbar.set_label('Loading Value', rotation=270, labelpad=20)

# Add loading values as text
for i in range(2):
    for j in range(len(selected_features)):
        text = axes[1, 1].text(j, i, f'{loadings_df.iloc[j, i]:.2f}',
                              ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

# Print detailed loading information
print("=" * 70)
print("PCA ANALYSIS SUMMARY")
print("=" * 70)
print(f"\nExplained Variance Ratio:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.4f} ({var*100:.2f}%)")

print(f"\nCumulative Variance: {cumsum}")

print("\n" + "=" * 70)
print("FEATURE LOADINGS (CONTRIBUTIONS)")
print("=" * 70)

# Create detailed loadings dataframe
all_loadings_df = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(pca.components_))],
    index=selected_features
)

print("\nLoadings for all Principal Components:")
print(all_loadings_df.round(4))

# Show which features contribute most to each PC
print("\n" + "=" * 70)
print("TOP CONTRIBUTING FEATURES PER COMPONENT")
print("=" * 70)
for i in range(min(3, len(pca.components_))):
    print(f"\nPC{i+1} (explains {pca.explained_variance_ratio_[i]*100:.2f}% of variance):")
    abs_loadings = abs(all_loadings_df[f'PC{i+1}'])
    top_features = abs_loadings.sort_values(ascending=False)
    for feature, loading in top_features.items():
        actual_loading = all_loadings_df.loc[feature, f'PC{i+1}']
        print(f"  {feature}: {actual_loading:+.4f} (|{loading:.4f}|)")